# SmartCare Disease Risk Classification

## AI-Based Patient Disease Risk Classification

- Module: CCS3440 – Artificial Intelligence
- Option C – Disease Risk Classification
- Target: disease_risk_level
- Problem Type: Multi-Class Classification
- Classes: Low, Medium, High

## 1. Project Setup

In [ ]:
# Imports and configuration
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance
import joblib

RANDOM_STATE = 42
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10,6)

## 2. Dataset Loading

In [ ]:
# Find project root (not hardcoding absolute path) and load CSVs
def find_project_root(start=Path.cwd()):
    p = Path(start).resolve()
    for _ in range(6):
        if (p / 'data' / 'raw').exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError('Project root with data/raw not found')

PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_FILE = DATA_RAW / 'smartcare_ai_dataset_1000.csv'
DICT_FILE = DATA_RAW / 'smartcare_ai_dataset_data_dictionary.csv'

for p in (DATA_FILE, DICT_FILE):
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

df = pd.read_csv(DATA_FILE)
data_dictionary = pd.read_csv(DICT_FILE)
df_raw = df.copy(deep=True)
print('Loaded', DATA_FILE.name, 'with shape', df.shape)
print('Loaded data dictionary with shape', data_dictionary.shape)

# Task 02 – Dataset Understanding

## 2.1 Dataset Overview

In [ ]:
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
df.head()

## 2.2 Data Dictionary

In [ ]:
data_dictionary.head(50)

## 2.3 Data Types

In [ ]:
df.info()
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object','category']).columns.tolist()
possible_date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
numeric_cols, categorical_cols, possible_date_cols

## 2.4 Target Variable

In [ ]:
TARGET = 'disease_risk_level'
assert TARGET in df.columns, f'Missing target {TARGET}'
print('Unique classes:', df[TARGET].unique())
display(df[TARGET].value_counts())
display(df[TARGET].value_counts(normalize=True).mul(100).round(2))
print('Missing target values:', df[TARGET].isna().sum())

## 2.5 Missing Values

In [ ]:
missing = df.isna().sum().rename('missing_count').reset_index().rename(columns={'index':'column'})
missing['missing_pct'] = (missing['missing_count']/len(df)*100).round(2)
missing.sort_values('missing_count', ascending=False).reset_index(drop=True)

## 2.6 Duplicate Records

In [ ]:
print('Full-row duplicates:', df.duplicated().sum())
for c in ['record_id','patient_id']:
    if c in df.columns:
        print(c, 'unique / total:', df[c].nunique(), '/', len(df))

## 2.7 Unique Values / Cardinality

In [ ]:
card = []
for c in categorical_cols:
    card.append({'column': c, 'n_unique': df[c].nunique(), 'sample_vals': df[c].dropna().unique()[:10]})
pd.DataFrame(card).sort_values('n_unique', ascending=False).reset_index(drop=True)

## 2.8 Descriptive Statistics

In [ ]:
display(df.describe(include='all').T)
display(df.describe())

# Target leakage and feature audit

In [ ]:
LEAKAGE_COLUMNS = [c for c in ['no_show','readmitted_30_days'] if c in df.columns]
IDENTIFIER_COLUMNS = [c for c in ['record_id','patient_id'] if c in df.columns]
print('Leakage columns:', LEAKAGE_COLUMNS)
print('Identifier columns:', IDENTIFIER_COLUMNS)

# Task 03 – Data Preprocessing and Feature Engineering

## 3.1 Working data copy

In [ ]:
df_work = df.copy(deep=True)
df_work.shape

## 3.2 Duplicate Handling

In [ ]:
before = df_work.shape[0]
n_dups = df_work.duplicated().sum()
if n_dups>0:
    df_work = df_work.drop_duplicates().reset_index(drop=True)
after = df_work.shape[0]
print('Duplicates removed:', before-after)

## 3.3 Missing Value Strategy
Imputation will be applied inside pipelines: numeric -> median, categorical -> most frequent.

## 3.4 Outlier Identification

In [ ]:
outliers = []
for c in numeric_cols:
    q1 = df_work[c].quantile(0.25)
    q3 = df_work[c].quantile(0.75)
    iqr = q3-q1
    lower = q1-1.5*iqr
    upper = q3+1.5*iqr
    n = df_work[(df_work[c]<lower)|(df_work[c]>upper)].shape[0]
    outliers.append({'feature':c,'n_outliers':int(n)})
pd.DataFrame(outliers).sort_values('n_outliers', ascending=False).head(10)

## 3.5 Data Type Cleaning

In [ ]:
for c in possible_date_cols:
    df_work[c] = pd.to_datetime(df_work[c], errors='coerce')
possible_date_cols

## 3.6 Feature Engineering

In [ ]:
engineered = []
if set(['systolic_bp','diastolic_bp']).issubset(df_work.columns):
    df_work['pulse_pressure'] = df_work['systolic_bp'] - df_work['diastolic_bp']
    engineered.append('pulse_pressure')
engineered

## 3.7 Feature Selection

In [ ]:
drop_cols = [c for c in [TARGET] + LEAKAGE_COLUMNS + IDENTIFIER_COLUMNS if c in df_work.columns]
X_cols = [c for c in df_work.columns if c not in drop_cols]
print('Dropping from predictors:', drop_cols)
print('Number of predictors:', len(X_cols))

## 3.8 Input / Target Creation

In [ ]:
X = df_work[X_cols].copy()
y = df_work[TARGET].copy()
X.shape, y.shape

## 3.9 Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
X_train.shape, X_test.shape